Create a table of original and revival eyeshadow products.
*Co-authored with CoCo*

In [ ]:
%%sql -r dataframe_1
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r dataframe_2
SELECT
  era,
  product_name,
  product_type,
  shade_name,
  finish,
  finish_group,
  c_pct,
  m_pct,
  y_pct,
  k_pct
FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES
WHERE product_type ILIKE '%Eyeshadow%'
  AND shade_name IS NOT NULL
ORDER BY era, product_name, shade_name;

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
%%sql -r eyeshadow_scatter_data
WITH eyeshadow_scatter AS (
  SELECT
    era,
    product_name,
    product_type,
    shade_name,
    finish,
    finish_group,
    c_pct,
    m_pct,
    y_pct,
    k_pct,
    ROUND(k_pct + 0.35 * m_pct + 0.15 * y_pct, 2) AS depth_score,
    ROUND((y_pct - m_pct) - 0.5 * c_pct, 2) AS undertone_score
  FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES
  WHERE product_type ILIKE '%Eyeshadow%'
    AND shade_name IS NOT NULL
    AND c_pct IS NOT NULL
    AND m_pct IS NOT NULL
    AND y_pct IS NOT NULL
    AND k_pct IS NOT NULL
)
SELECT
  era,
  product_name,
  product_type,
  shade_name,
  finish,
  finish_group,
  c_pct,
  m_pct,
  y_pct,
  k_pct,
  depth_score,
  undertone_score
FROM eyeshadow_scatter
ORDER BY era, product_name, shade_name, finish;

In [ ]:
import pandas as pd
from matplotlib.lines import Line2D

def cmyk_to_rgb(c, m, y, k):
    c, m, y, k = c / 100, m / 100, y / 100, k / 100
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return (r, g, b)

def map_finish_family(finish):
    if finish == 'Matte':
        return 'matte'
    if finish in {'Satin', 'Sheen', 'Pearl'}:
        return 'soft'
    if finish in {'Frost', 'Glitter', 'Metallic', 'Shimmer', 'Duochrome'}:
        return 'shiny'
    return 'other'

df = eyeshadow_scatter_data.to_pandas() if hasattr(eyeshadow_scatter_data, 'to_pandas') else eyeshadow_scatter_data
df = df.copy()
df.columns = [col.lower() for col in df.columns]
df['rgb'] = df.apply(lambda row: cmyk_to_rgb(row['c_pct'], row['m_pct'], row['y_pct'], row['k_pct']), axis=1)
df['finish_family_simple'] = df['finish'].apply(map_finish_family)

finish_markers = {
    'matte': 's',
    'soft': 'o',
    'shiny': '*',
    'other': '^',
}

era_order = ['original_launch', 'revival']
era_labels = {'original_launch': '2013 Collection', 'revival': '2026 Revival'}
era_annotations = {
    'original_launch': 'Palette-based neutrals in matte and satin finishes',
    'revival': 'Singles in bold, shiny finishes across a wider color range',
}

def get_extreme_labels(era_df, n=4):
    extremes = set()
    if len(era_df) <= n * 2:
        return set(era_df['shade_name'])
    extremes.update(era_df.nsmallest(n, 'depth_score')['shade_name'])
    extremes.update(era_df.nlargest(n, 'depth_score')['shade_name'])
    extremes.update(era_df.nsmallest(2, 'undertone_score')['shade_name'])
    extremes.update(era_df.nlargest(2, 'undertone_score')['shade_name'])
    return extremes

def plot_eyeshadow(layout='horizontal'):
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
        order = era_order
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14), sharex=True)
        order = era_order

    for i, (axis, era) in enumerate(zip(axes.flat, order)):
        era_df = df[df['era'] == era].copy()
        axis.axhline(0, color='#d9d9d9', linewidth=1, zorder=0)

        for finish_family_simple, finish_df in era_df.groupby('finish_family_simple'):
            colors = list(finish_df['rgb'])
            axis.scatter(
                finish_df['depth_score'],
                finish_df['undertone_score'],
                s=130,
                c=colors,
                marker=finish_markers.get(finish_family_simple, '^'),
                edgecolors='black',
                linewidths=0.6,
                alpha=0.95,
                label=finish_family_simple.title(),
            )

        extremes = get_extreme_labels(era_df)
        for _, row in era_df.iterrows():
            if row['shade_name'] in extremes:
                axis.annotate(
                    row['shade_name'],
                    (row['depth_score'], row['undertone_score']),
                    textcoords='offset points',
                    xytext=(5, 4),
                    fontsize=9,
                    fontweight='normal',
                    alpha=0.85,
                )

        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, era_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('Shade depth, from lighter to deeper', fontsize=11, fontweight='normal')
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)

        if layout == 'vertical' or i == 0:
            axis.set_ylabel('Undertone, from cool to warm', fontsize=11, fontweight='normal')

    legend_handles = [
        Line2D([0], [0], marker=marker, color='black', markerfacecolor='white',
               linestyle='', markersize=9, label=name.title())
        for name, marker in finish_markers.items()
        if name in df['finish_family_simple'].unique()
    ]
    if layout == 'horizontal':
        axes.flat[0].legend(handles=legend_handles, title='Finish family',
                            loc='lower left', bbox_to_anchor=(0, 1.15),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        fig.suptitle('EYESHADOW SHIFTS FROM NEUTRAL PALETTES TO BOLD SINGLES',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        axes.flat[0].legend(handles=legend_handles, title='Finish family',
                            loc='lower left', bbox_to_anchor=(0, 1.08),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('EYESHADOW SHIFTS FROM NEUTRAL PALETTES TO BOLD SINGLES',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_eyeshadow('horizontal')
plot_eyeshadow('vertical')